# CNN 모델

이 노트북은 `src/models/cnn.py`의 `CNN`을 직접 실행하고 forward/backward 흐름, CuPy/numpy 경계, MLP와의 인터페이스 비교를 검증하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (CuPy 없으면 numpy fallback 자동 적용)

**목표**
- `CNN` 아키텍처 구성과 파라미터 수를 확인한다.
- forward에서 `(N, 784)` numpy 입력이 CuPy Conv 경로를 거쳐 numpy logit으로 반환되는 흐름을 확인한다.
- backward에서 numpy FC gradient가 CuPy Conv 경로로 역전파되는 흐름을 확인한다.
- MLP와 동일한 인터페이스로 학습 루프를 실행할 수 있음을 확인한다.
- train/eval 모드 전환과 Dropout 동작 차이를 확인한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# CuPy 가용 여부 확인
try:
    import cupy as cp
    print(f"CuPy available: {cp.__version__}")
    CUPY_AVAILABLE = True
except ImportError:
    print("CuPy not available → CNN runs on numpy (CPU fallback)")
    CUPY_AVAILABLE = False

In [ ]:
from src.models.cnn import CNN
from src.models.mlp import MLP
from src.nn.losses import (
    cross_entropy, cross_entropy_grad,
    binary_cross_entropy, binary_cross_entropy_grad,
    mse, mse_grad,
)
from src.nn.metrics import accuracy

## 1. 개요

`CNN`은 CuPy 기반 2층 합성곱 네트워크이다.
`MLP`와 동일한 `forward`/`backward`/`params`/`grads` 인터페이스를 유지하므로
Trainer, Evaluator, Predictor가 모델 종류를 구분하지 않고 동일하게 동작한다.

**구조**:

```
입력: (N, 784) numpy
  reshape → (N, 1, 28, 28) CuPy
  [Conv 블록 1] Conv2d(1,32,3,pad=1) → ReLU → MaxPool2d(2,2)  → (N, 32, 14, 14)
  [Conv 블록 2] Conv2d(32,64,3,pad=1) → ReLU → MaxPool2d(2,2) → (N, 64, 7, 7)
  Flatten                                                       → (N, 3136) CuPy
  .get() ── CuPy/numpy 경계 ──────────────────────────────────  → (N, 3136) numpy
  [FC 블록]  Dropout(0.5) → Linear(3136,256) → ReLU → Linear(256,output_dim)
출력: (N, output_dim) numpy  ← raw logit
```

| 블록 | 구성 | 역할 |
|---|---|---|
| Conv 블록 1 | `Conv2d(1,32) → ReLU → MaxPool2d(2,2)` | 저수준 특징(선, 모서리) 추출 |
| Conv 블록 2 | `Conv2d(32,64) → ReLU → MaxPool2d(2,2)` | 고수준 특징(곡선, 부분 형태) 추출 |
| FC 블록 | `Dropout → Linear(3136,256) → ReLU → Linear(256,output)` | 특징 조합 및 분류 |

## 2. 모델 생성과 파라미터 구성

`CNN.params = [conv1_W, conv1_b, conv2_W, conv2_b, fc1_W, fc1_b, fc2_W, fc2_b]` (총 8개 배열)

앞 4개는 CuPy(또는 numpy fallback) 배열, 뒤 4개는 numpy 배열이다.

In [ ]:
model = CNN(task="multiclass", seed=42)

print(f"task:       {model.task}")
print(f"output_dim: {model.output_dim}")
print(f"params 수:  {len(model.params)}  <- Conv 2×(W,b) + FC 2×(W,b) = 8")
print()
param_names = ["conv1_W", "conv1_b", "conv2_W", "conv2_b",
               "fc1_W",   "fc1_b",   "fc2_W",   "fc2_b"]
print(f"{'이름':<12} {'shape':<22} {'dtype':<10} {'배열 타입'}")
print("-" * 65)
for name, p in zip(param_names, model.params):
    arr_type = type(p).__module__.split('.')[0]  # 'cupy' or 'numpy'
    print(f"{name:<12} {str(p.shape):<22} {str(p.dtype):<10} {arr_type}")

In [ ]:
# 총 파라미터 수 계산
total = sum(p.size for p in model.params)
print(f"총 파라미터 수: {total:,}")
print(f"메모리 (float32): {total * 4 / 1024**2:.2f} MB")
print()

# 레이어별 분해
layer_names = [("Conv2d(1→32,K=3)",    0), ("Conv2d(32→64,K=3)", 2),
               ("Linear(3136→256)",     4), ("Linear(256→output)", 6)]
print(f"{'레이어':<25} {'파라미터 수'}")
print("-" * 40)
for lname, i in layer_names:
    w, b = model.params[i], model.params[i+1]
    count = w.size + b.size
    print(f"{lname:<25} {count:>10,}")
print(f"{'합계':<25} {total:>10,}")

## 3. Forward — (N, 784) numpy → (N, output_dim) numpy

입력은 항상 `(N, 784)` numpy float32이고, 출력은 항상 `(N, output_dim)` numpy float32이다.
내부에서 CuPy 변환과 역변환이 자동으로 처리된다.

In [ ]:
N = 4  # CNN은 im2col 연산이 있어 MLP보다 느리므로 소규모 배치로 확인
x = np.random.randn(N, 784).astype(np.float32)

for task, expected_dim in [("multiclass", 10), ("binary", 1), ("regression", 1)]:
    m = CNN(task=task, seed=0)
    m.eval()  # Dropout 비활성화
    logits = m.forward(x)
    arr_type = type(logits).__module__.split('.')[0]
    print(f"task={task:<12}: logits.shape={logits.shape}, dtype={logits.dtype}, 배열={arr_type}")
    assert logits.shape == (N, expected_dim)
    assert logits.dtype == np.float32
    assert isinstance(logits, np.ndarray), "output must be numpy array"

## 4. CuPy/numpy 경계

forward에서 `Flatten` 이후 `.get()`으로 CuPy → numpy 변환이 일어난다.
backward에서 numpy FC gradient를 `_xp.asarray()`로 CuPy로 변환하여 Conv 경로에 전달한다.

```
forward:  ... → Flatten (CuPy) --.get()--> Dropout (numpy) → ...
backward: ... ← Flatten (CuPy) ←-_xp.asarray()-- Dropout (numpy) ← ...
```

In [ ]:
model = CNN(task="multiclass", seed=42)
model.eval()

N = 4
x = np.random.randn(N, 784).astype(np.float32)
print(f"입력 배열 타입: {type(x).__module__}")

logits = model.forward(x)
print(f"출력 배열 타입: {type(logits).__module__}")
print(f"출력 shape: {logits.shape}")
print(f"출력 dtype: {logits.dtype}")
print()
print("→ 입출력 모두 numpy float32")
print("  내부에서 CuPy 변환(Flatten 전까지)과 역변환(Flatten 이후)이 자동으로 처리됨")

## 5. Backward — gradient 흐름

`model.backward(grad_out)`는 numpy grad를 FC 경로에서 역전파하고,
CuPy로 변환하여 Conv 경로를 역전파한다.

In [ ]:
model = CNN(task="multiclass", seed=42)
model.train()

N = 4
x = np.random.randn(N, 784).astype(np.float32)
targets = np.eye(10, dtype=np.float32)[np.arange(N) % 10]

logits = model.forward(x)
loss = cross_entropy(logits, targets)
grad_out = cross_entropy_grad(logits, targets)

model.backward(grad_out)

print(f"loss: {loss:.4f}")
print(f"grad_out shape: {grad_out.shape}, type: {type(grad_out).__module__}")
print()
print("=== backward 후 gradient norm ===")
for name, g in zip(param_names, model.grads):
    arr_type = type(g).__module__.split('.')[0]
    # CuPy 배열은 float()로 스칼라 추출
    norm_val = float(np.linalg.norm(np.asarray(g)))
    print(f"  {name:<12}: norm={norm_val:.4f}  ({arr_type})")

## 6. train/eval 모드 — Dropout 동작 차이

`model.train()` / `model.eval()`은 `conv_net`, `dropout`, `fc_net` 각각에 모드를 전파한다.
학습 모드에서 Dropout은 FC 입력의 50%를 비활성화하고 나머지를 2배로 스케일링한다.
평가 모드에서는 입력을 그대로 통과시킨다.

In [ ]:
np.random.seed(0)
model = CNN(task="multiclass", seed=42)
N = 16
x = np.random.randn(N, 784).astype(np.float32)

# 학습 모드: 실행마다 Dropout 마스크가 달라져 출력이 달라진다
model.train()
out1 = model.forward(x)
out2 = model.forward(x)
print(f"train mode: 동일 입력에 대해 두 출력 동일 여부: {np.allclose(out1, out2)}")
print(f"  (Dropout 때문에 False 기대)")

# 평가 모드: Dropout 없으므로 동일 입력 → 동일 출력
model.eval()
out3 = model.forward(x)
out4 = model.forward(x)
print(f"\neval mode:  동일 입력에 대해 두 출력 동일 여부: {np.allclose(out3, out4)}")
print(f"  (Dropout 없으므로 True 기대)")

## 7. MLP vs CNN 인터페이스 비교

두 모델은 동일한 인터페이스를 제공하므로 Trainer는 모델 종류를 구분하지 않고 동일한 코드로 동작한다.

In [ ]:
N = 4
x = np.random.randn(N, 784).astype(np.float32)
targets = np.eye(10, dtype=np.float32)[np.arange(N) % 10]

print(f"{'항목':<22} {'MLP':<22} {'CNN'}")
print("-" * 65)

for ModelClass in [MLP, CNN]:
    m = ModelClass(task="multiclass", seed=42)
    m.eval() if hasattr(m, 'eval') else None
    logits = m.forward(x)
    loss = cross_entropy(logits, targets)
    grad_out = cross_entropy_grad(logits, targets)
    m.backward(grad_out)
    name = ModelClass.__name__
    n_params = len(m.params)
    logit_type = type(logits).__module__.split('.')[0]
    logit_shape = str(logits.shape)
    print(f"  [{name}]")
    print(f"  {'forward 출력':<20} {logit_shape}  ({logit_type})")
    print(f"  {'params 수':<20} {n_params}")
    print(f"  {'loss':<20} {loss:.4f}")
    print()

## 8. 수동 SGD 학습 — MLP vs CNN 비교

소규모 synthetic 데이터로 두 모델을 동일한 학습 루프로 실행하여 loss 감소를 비교한다.

In [ ]:
np.random.seed(0)
N_train = 16
n_steps = 30
lr = 0.01

x_syn = np.random.randn(N_train, 784).astype(np.float32)
y_syn = np.eye(10, dtype=np.float32)[np.arange(N_train) % 10]

results = {}
for ModelClass in [MLP, CNN]:
    m = ModelClass(task="multiclass", seed=42)
    m.train() if hasattr(m, 'train') else None
    losses = []
    for step in range(n_steps):
        logits = m.forward(x_syn)
        losses.append(float(cross_entropy(logits, y_syn)))
        grad = cross_entropy_grad(logits, y_syn)
        m.backward(grad)
        for p, g in zip(m.params, m.grads):
            p -= lr * np.asarray(g)  # CuPy 배열도 numpy로 변환하여 처리
    results[ModelClass.__name__] = losses
    print(f"{ModelClass.__name__:<5}: loss {losses[0]:.3f} → {losses[-1]:.3f}")

plt.figure(figsize=(7, 4))
for name, losses in results.items():
    plt.plot(losses, linewidth=2, label=name)
plt.title("MLP vs CNN — Cross Entropy Loss (synthetic data)")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. 공간 크기 변화 추적

입력 `(N, 1, 28, 28)`부터 Flatten까지 각 레이어를 통과하는 shape를 확인한다.

In [ ]:
try:
    import cupy as _xp
except ImportError:
    import numpy as _xp

from src.nn.conv import Conv2d, MaxPool2d, Flatten
from src.nn.layers import ReLU

N = 2
x_img = _xp.asarray(np.random.randn(N, 1, 28, 28).astype(np.float32))

conv_layers = [
    ("reshape → (N,1,28,28)",        None),
    ("Conv2d(1,32,K=3,pad=1)",        Conv2d(1, 32, 3, padding=1, seed=0, xp=_xp)),
    ("ReLU",                           ReLU()),
    ("MaxPool2d(K=2)",                 MaxPool2d(2, 2, xp=_xp)),
    ("Conv2d(32,64,K=3,pad=1)",        Conv2d(32, 64, 3, padding=1, seed=1, xp=_xp)),
    ("ReLU",                           ReLU()),
    ("MaxPool2d(K=2)",                 MaxPool2d(2, 2, xp=_xp)),
    ("Flatten",                        Flatten()),
]

print(f"{'레이어':<30} {'Input shape':<22} {'Output shape'}")
print("-" * 72)
h = x_img
print(f"{'(input)':<30} {'-':<22} {str(h.shape)}")
for name, layer in conv_layers:
    if layer is None:
        continue
    in_shape = h.shape
    h = layer.forward(h)
    print(f"{name:<30} {str(in_shape):<22} {str(h.shape)}")

print(f"\n→ Flatten 출력 {h.shape[1]} = 64 × 7 × 7")

## 10. 검증

In [ ]:
N = 4
x_t = np.random.randn(N, 784).astype(np.float32)

# task별 forward shape
for task, exp_dim in [("multiclass", 10), ("binary", 1), ("regression", 1)]:
    m = CNN(task=task, seed=0)
    m.eval()
    logits = m.forward(x_t)
    assert logits.shape == (N, exp_dim), f"{task}: {logits.shape}"
    assert logits.dtype == np.float32
    assert isinstance(logits, np.ndarray), "output must be numpy array"

# params/grads 구조
m = CNN(task="multiclass", seed=0)
assert len(m.params) == 8
assert len(m.grads) == 8
assert m.params[0].shape == (32, 1, 3, 3)    # conv1_W
assert m.params[2].shape == (64, 32, 3, 3)   # conv2_W
assert m.params[4].shape == (3136, 256)       # fc1_W
assert m.params[6].shape == (256, 10)         # fc2_W

# backward 후 grads 비 0
m.train()
t_t = np.eye(10, dtype=np.float32)[np.arange(N) % 10]
logits = m.forward(x_t)
m.backward(cross_entropy_grad(logits, t_t))
for name, g in zip(param_names, m.grads):
    g_np = np.asarray(g)
    assert np.any(g_np != 0), f"{name} grads must be non-zero"

# eval 모드: 동일 입력 → 동일 출력
m.eval()
o1 = m.forward(x_t)
o2 = m.forward(x_t)
assert np.allclose(o1, o2), "eval mode must produce deterministic output"

# train 모드: 동일 입력 → 다를 수 있음 (Dropout)
m.train()
np.random.seed(None)  # seed 고정 해제
o3 = m.forward(x_t)
o4 = m.forward(x_t)
# Dropout으로 인해 다를 가능성 높음 (확률적이므로 assert 불가, 확인만)
print(f"train mode 두 출력 동일: {np.allclose(o3, o4)}  (Dropout → 보통 False)")
print()
print("all CNN validation passed")

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 항목 | 내용 |
|---|---|
| 구조 | Conv2블록 + FC블록 (Dropout 포함) |
| 총 파라미터 수 | 824,458 (multiclass 기준), float32 ≈ 3.1 MB |
| `params` | 8개 배열 (Conv 4개 CuPy + FC 4개 numpy) |
| `forward` 입출력 | `(N, 784)` numpy → `(N, output_dim)` numpy |
| CuPy/numpy 경계 | Flatten 이후 `.get()` (forward) / `_xp.asarray()` (backward) |
| Dropout | `train()` 시 50% 비활성화 (inverted scaling), `eval()` 시 통과 |

**MLP vs CNN 인터페이스 비교**

| 항목 | MLP | CNN |
|---|---|---|
| 입력 | `(N, 784)` numpy | `(N, 784)` numpy |
| 출력 | `(N, output_dim)` numpy | `(N, output_dim)` numpy |
| params 수 | 6개 (모두 numpy) | 8개 (CuPy 4 + numpy 4) |
| GPU 실행 | 없음 | Conv 경로 (CuPy) |
| Dropout | 없음 | FC 첫 레이어 전 |

**핵심 설계 원칙**
- 두 모델의 입출력이 동일하므로 Trainer, Evaluator, Predictor는 모델 종류를 구분하지 않고 동일한 코드로 동작한다.
- CuPy 미설치 환경에서 자동으로 numpy fallback이 적용되어 동일한 코드로 기능 검증이 가능하다.